In [2]:
%run "hw.ipynb"

In [3]:
import rdata
import xarray as xr

# Load the Rdata file containing EURO-CORDEX output for the Uccle (Brussels) gridpoint
parsed = rdata.read_rda('./data/euro_cordex_daily_tasmin_tasmax_tas_ukkel.Rdata')

# Selecting simulations

In [5]:
from scipy.stats import ks_2samp
import numpy as np
from itertools import product


# Load historical temperature data
year, month, day, Tmax, Tmin = np.loadtxt("./data/data_1878_2023.txt", unpack=True)
dataset_his = Dataset(year, month, day, Tmax, Tmin)
dataset_his.set_all()


# Load EURO-CORDEX output evaluated at Uccle (Brussels) gridpoint
tasmax = parsed['arr.loc.av.tasmax.ukkel'] - 273.15 # Rescale K to °C
tasmin = parsed['arr.loc.av.tasmin.ukkel'] - 273.15

parsed_his = tasmax.sel(exp='historical')
parsed_rcp = tasmax.sel(exp='rcp85')


# Step 1: Valid year ranges per run

# We assume that if the data for January 1st is present, then the entire year is present (no NaNs)
valid_years_per_run = {}
for run in parsed_his['run'].values:
    jan_first = parsed_his.sel(run=run, month='Jan', day='1')
    valid_years = [y for y in jan_first['year'].values if not np.isnan(jan_first.sel(year=y).values)]
    if valid_years:
        # Historical comparison window: 1971 .. last valid historical year for that historical run
        valid_years_per_run[run] = (1971, int(max(valid_years)))

# First valid rcp85 year per run
first_valid_rcp85_years = {}
for run in parsed_rcp['run'].values:
    jan_first = parsed_rcp.sel(run=run, month='Jan', day='1')
    valid_years = [y for y in jan_first['year'].values if not np.isnan(jan_first.sel(year=y).values)]
    if valid_years:
        first_valid_rcp85_years[run] = int(min(valid_years))


# Step 2: KS tests per run

# Set the match failure thresholds
KS_THRESH_CDD  = 0.264
KS_THRESH_TMAX = 0.100
KS_THRESH_TMIN = 0.100

passed = []  # collect (run, ks_cdd, ks_tmax, ks_tmin)
passed_models = []

for run in tasmax['run'].values:
    if run not in valid_years_per_run or run not in first_valid_rcp85_years:
        continue # Skip runs that have no valid years in historical OR RCP8.5 run

    year_start_his, year_end_his = valid_years_per_run[run]
    years_his = [str(y) for y in range(year_start_his, year_end_his + 1)]
    years_rcp = [str(y) for y in range(first_valid_rcp85_years[run], 2024)]

    # Select temperature data (historical + rcp85)
    # Years not covered are simply skipped / left out
    Tmax_his = tasmax.sel(run=run, exp='historical', year=years_his)
    Tmax_rcp = tasmax.sel(run=run, exp='rcp85',      year=years_rcp)
    Tmin_his = tasmin.sel(run=run, exp='historical', year=years_his)
    Tmin_rcp = tasmin.sel(run=run, exp='rcp85',      year=years_rcp)

    # Combine sequences and flatten to lists
    Tmax_s = list(Tmax_his.values.flatten()) + list(Tmax_rcp.values.flatten())
    Tmin_s = list(Tmin_his.values.flatten()) + list(Tmin_rcp.values.flatten())

    # Build matching date vectors for Dataset (year, month, day)
    years_all = [int(y) for y in years_his] + [int(y) for y in years_rcp]
    months = list(range(1, 13))
    days = list(range(1, 32))

    # Repeat year per (12*31) days, and tile months/days across years
    year_s = np.repeat(years_all, len(months) * len(days)).tolist()
    month_s = np.tile(np.repeat(months, len(days)), len(years_all)).tolist()
    day_s = np.tile(days, len(years_all) * len(months)).tolist()

    # Construct Dataset instance containing the simulation output
    dataset_s = Dataset(year_s, month_s, day_s, Tmax_s, Tmin_s)
    dataset_s.set_all()


    # Daily series historical slice: from first day of year_start_his to end of dataset (2023)
    i_start_day = list(dataset_his.year).index(year_start_his)
    historical_Tmax_daily = dataset_his.Tmax[i_start_day: -1]
    historical_Tmin_daily = dataset_his.Tmin[i_start_day: -1]

    # Yearly CDD list over all considered years (his + rcp years limited to 2023)
    years_all_limited = [y for y in years_all if y <= 2023]
    historical_CDDs = [dataset_his.CDD[y] for y in years_all_limited]

    simulated_CDDs = list(dataset_s.CDD.values())
    simulated_Tmax_daily = dataset_s.Tmax
    simulated_Tmin_daily = dataset_s.Tmin

    # Perform the KS tests
    ks_cdd, _  = ks_2samp(simulated_CDDs, historical_CDDs)
    ks_tmax, _ = ks_2samp(simulated_Tmax_daily, historical_Tmax_daily)
    ks_tmin, _ = ks_2samp(simulated_Tmin_daily, historical_Tmin_daily)

    # Collect models passing all tests
    if ks_cdd <= KS_THRESH_CDD and ks_tmax <= KS_THRESH_TMAX and ks_tmin <= KS_THRESH_TMIN:
        passed.append((run, ks_cdd, ks_tmax, ks_tmin))
        passed_models.append(run)

# Summary output
print("Models passing KS testing:")
if not passed:
    print("  None")
else:
    for run, ks_cdd, ks_tmax, ks_tmin in passed:
        print(f"  {run}: CDD={ks_cdd:.3f}, Tmax={ks_tmax:.3f}, Tmin={ks_tmin:.3f}")


Models passing KS testing:
  MOHC-HadGEM2-ES_r1i1p1_HIRHAM5_v1_EUR-11: CDD=0.245, Tmax=0.059, Tmin=0.075
  MPI-M-MPI-ESM-LR_r3i1p1_REMO2015_v1_EUR-11: CDD=0.170, Tmax=0.039, Tmin=0.042
  NCC-NorESM1-M_r1i1p1_REMO2015_v1_EUR-11: CDD=0.245, Tmax=0.026, Tmin=0.088
  MOHC-HadGEM2-ES_r1i1p1_RACMO22E_v2_EUR-11: CDD=0.151, Tmax=0.049, Tmin=0.089
  MOHC-HadGEM2-ES_r1i1p1_HadREM3-GA7-05_v1_EUR-11: CDD=0.226, Tmax=0.048, Tmin=0.049
  MPI-M-MPI-ESM-LR_r1i1p1_REMO2009_v1_EUR-11: CDD=0.151, Tmax=0.045, Tmin=0.048
  MPI-M-MPI-ESM-LR_r2i1p1_REMO2009_v1_EUR-11: CDD=0.113, Tmax=0.041, Tmin=0.044
  MPI-M-MPI-ESM-LR_r1i1p1_REMO2015_v1_EUR-22: CDD=0.132, Tmax=0.040, Tmin=0.056
  NCC-NorESM1-M_r1i1p1_REMO2015_v1_EUR-22: CDD=0.245, Tmax=0.026, Tmin=0.100
  MOHC-HadGEM2-ES_r1i1p1_CCLM5-0-6_v1_EUR-44: CDD=0.208, Tmax=0.093, Tmin=0.040
  MPI-M-MPI-ESM-LR_r2i1p1_REMO2009_v1_EUR-44: CDD=0.231, Tmax=0.064, Tmin=0.073
  CCCma-CanESM2_r1i1p1_RCA4_v1_EUR-44: CDD=0.212, Tmax=0.071, Tmin=0.051
  MIROC-MIROC5_r1i1p1_RC

# Mining years of extreme heat

In [9]:
def rate_event(ds, yr):
    """Function to rate an event (yr) 1 if its indices are in the GWS target ranges or 0 if they are not
    Parameters
    ----------
    ds
        Dataset instance containing the climate simulation projection from which the event is mined
    yr
        integer representing the event (year) itself
    Returns
    -------
        1 if the HI and Tmax are withing the scaling target 5% ranges (indicating a valid year of extreme heat in the 
        climate under study) and 0 if they are not
    """
    
    if yr <= 2025: # Only mine future years
        return 0
    
    yr_index = ds.yearset.index(yr)
    
    if ds.CDD[yr] >= 393.73 and ds.CDD[yr] <= 435.17:
        pass
    else:
        return 0
    
    if ds.Tmax_year[yr_index] >= 38.94 and ds.Tmax_year[yr_index] <= 43.04:
        return 1
    return 0

In [11]:
from datetime import datetime

class Match:
    """Class used to represent GWS scaling matches or hits, ie. valid years of extreme heat
    Attributes
    ----------
    yr
        integer representing the event (year) itself
    run
        string containing the model information for the RCP8.5 run under analysis,
        of the format <GCM>_<member>_<RCM>_<domain>
    ds
        Dataset instance containing the climate simulation projection from which the event is mined
    CDD, Tmax
        the climate indices HI and Tmax (floats) associated with the event
    Methods
    -------
    write()
    plot_year(ax)
    """
    def __init__(self, year, score, dataset, run):
        """Constructor"""
        self.yr = year
        self.run = run
        self.ds = dataset
        self.CDD = dataset.CDD[year]
        yr_index = dataset.yearset.index(year)
        self.Tmax = dataset.Tmax_year[yr_index]


    def write(self):
        """Function for outputting relevant information on the event"""
        runlen = len(self.run)
        print('-' * runlen)
        print(self.run)
        print('-' * runlen)
        print(self.yr)
        print(f'CDD = {self.CDD}, Tmax = {self.Tmax}°C')
        
    def plot_year(self, ax):
        """Function for plotting out TX/TN series as well as heatwave periods (RMI definition) on specified Axes object"""
        i_start = list(self.ds.year).index(self.yr)
        i_end = list(self.ds.year).index(self.yr + 1)

        ds_ = Dataset(
            self.ds.year[i_start:i_end], 
            self.ds.month[i_start:i_end], 
            self.ds.day[i_start:i_end], 
            self.ds.Tmax[i_start:i_end], 
            self.ds.Tmin[i_start:i_end]
        )
        
        # Convert string dates to datetime objects (assumes format 'd/m')
        dates = [datetime.strptime(ds_.get_datestring(index) + f'/{self.yr}', '%d/%m/%Y') for index in range(len(ds_.Tmax))]

        # Plot daily temperatures
        ax.plot(dates, ds_.Tmax, label='TX', color='red')
        ax.plot(dates, ds_.Tmin, label='TN', color='blue')
    
        # Plot heatwave periods
        hw_ranges = ds_.find_heatwaves(self.yr)
        for hw in hw_ranges:
            first_date = datetime.strptime(ds_.get_datestring(hw[0]) + f'/{self.yr}', '%d/%m/%Y')
            last_date = datetime.strptime(ds_.get_datestring(hw[1]+1) + f'/{self.yr}', '%d/%m/%Y')
            ax.axvspan(first_date, last_date, facecolor='yellow', alpha=0.5)
    
        # Set monthly ticks and abbreviated labels
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    
        for label in ax.get_xticklabels(): # Tick formatting
            label.set_rotation(45)
            label.set_ha('center')
    
        ax.set(xlim=(dates[0], dates[-1]), ylim=(-5, 47), ylabel='Temp. (°C)')
        ax.set_yticks(range(-5, 50, 5))
        ax.grid(ls='--')
        ax.legend(fancybox=True, shadow=True)

In [13]:
matches = dict() # Initialize a dictionary that will relate runs to the GWS matches mined from their projections

s = []
for i in range(2006,2100):
    s.append(str(i))

for i, run in enumerate(passed_models):
    matches[run] = list() # The Match objects will be stored in this list

    # Collect TX/TN data for the Uccle (Brussels) gridpoint
    parsed_tasmax = parsed['arr.loc.av.tasmax.ukkel']
    Tmax_s = parsed_tasmax.sel(run=run, year=s, exp='rcp85') - 273.15
    parsed_tasmin = parsed['arr.loc.av.tasmin.ukkel']
    Tmin_s = parsed_tasmin.sel(run=run, year=s, exp='rcp85') - 273.15

    years = [int(y) for y in s]
    months = range(1,13)
    days = range(1, 32)

    year_s, month_s, day_s = [], [], []

    for i_yr in range(len(years)):
        for i_m in range(12):
            for i_d in range(31):
                year_s.append(years[i_yr])
                month_s.append(months[i_m])
                day_s.append(days[i_d])

    # Construct a Dataset instance using the simulation data
    dataset_s = Dataset(year_s, month_s, day_s, Tmax_s.values.flatten(), Tmin_s.values.flatten())
    dataset_s.set_all()

    # Mine years of extreme heat and collect them in the dictionary
    for yr in dataset_s.yearset:
        score = rate_event(dataset_s, yr)
        if score == 1:
            matches[run].append(Match(yr, score, dataset_s, run))


In [15]:
# Write out all matches
for run in passed_models:
    for match in matches[run]:
        match.write()

-----------------------------------------
MOHC-HadGEM2-ES_r1i1p1_RACMO22E_v2_EUR-11
-----------------------------------------
2059
CDD = 411.0551208496113, Tmax = 40.1238952636719°C
-----------------------------------------
MOHC-HadGEM2-ES_r1i1p1_RACMO22E_v2_EUR-11
-----------------------------------------
2061
CDD = 394.50953369140814, Tmax = 39.9089294433594°C
-----------------------------------------------
MOHC-HadGEM2-ES_r1i1p1_HadREM3-GA7-05_v1_EUR-11
-----------------------------------------------
2060
CDD = 432.56398315429897, Tmax = 42.48031005859377°C
-----------------------------------------------
MOHC-HadGEM2-ES_r1i1p1_HadREM3-GA7-05_v1_EUR-11
-----------------------------------------------
2084
CDD = 413.1332336425804, Tmax = 39.32265625000002°C
------------------------------------------
MOHC-HadGEM2-ES_r1i1p1_CCLM5-0-6_v1_EUR-44
------------------------------------------
2076
CDD = 400.182238769533, Tmax = 39.97905883789065°C
------------------------------------------
MOHC